In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm

# ============================================================
# Spin-9/2 operators
# ============================================================

j = 9 / 2

dim = int(2 * j + 1)

m_vals = np.arange(j, -j - 1, -1)

Jp = np.zeros((dim, dim), dtype=complex)
Jm = np.zeros((dim, dim), dtype=complex)
Jz = np.diag(m_vals)

for i, m in enumerate(m_vals[:-1]):
    coeff = np.sqrt(j * (j + 1) - m * (m - 1))
    Jm[i + 1, i] = coeff

Jp = Jm.conj().T

Jx = 0.5 * (Jp + Jm)
Jy = -0.5j * (Jp - Jm)

# ============================================================
# Initial spin coherent state along +z
# ============================================================

psi0 = np.zeros(dim, dtype=complex)
psi0[0] = 1.0

# ============================================================
# Simulation parameters
# ============================================================

Omega = 2.0
omega_x = 0.15
omega_y = -0.12
omega_z = 0.2

T = 20.0
dt = 0.02
N = int(T / dt)

time = np.arange(N) * dt

# Time-dependent phase

def phi(t):
    return 0.5 * np.sin(0.3 * t)

# ============================================================
# Exact quantum evolution
# ============================================================

psi = psi0.copy()

true_means = np.zeros((N, 3))
true_states = []

for k, t in enumerate(time):

    H = (
        Omega * (Jx * np.cos(phi(t)) + Jy * np.sin(phi(t)))
        + omega_x * Jx
        + omega_y * Jy
        + omega_z * Jz
    )

    U = expm(-1j * H * dt)
    psi = U @ psi

    psi = psi / np.linalg.norm(psi)

    true_states.append(psi.copy())

    true_means[k, 0] = np.real(np.vdot(psi, Jx @ psi))
    true_means[k, 1] = np.real(np.vdot(psi, Jy @ psi))
    true_means[k, 2] = np.real(np.vdot(psi, Jz @ psi))

# ============================================================
# Generate noisy measurements of Jz
# ============================================================

measurement_std = 0.4

measurements = (
    true_means[:, 2]
    + measurement_std * np.random.randn(N)
)

# ============================================================
# Linearized classical dynamics for mean spin vector
# ============================================================
#
# d<J>/dt = B_eff x <J>
#
# with
#
# B_eff = [
#   Omega cos(phi) + omega_x,
#   Omega sin(phi) + omega_y,
#   omega_z
# ]
#
# ============================================================

# ============================================================
# Augmented-state Kalman filter setup
# ============================================================
#
# State vector:
#
# x = [<Jx>, <Jy>, <Jz>, omega_x]
#
# We now jointly estimate the spin expectation values
# and the unknown parameter omega_x.
#
# ============================================================

# Initial estimate
x_est = np.array([
    0.0,
    0.0,
    j,
    0.0
])

# Initial covariance
P = np.diag([
    1.0,
    1.0,
    1.0,
    0.5
])

# Process noise
Q = np.diag([
    0.01,
    0.01,
    0.01,
    1e-5
])

# Measurement noise
R = np.array([[measurement_std**2]])

# Measurement matrix
H_meas = np.array([[0.0, 0.0, 1.0, 0.0]])

estimated_means = np.zeros((N, 3))
estimated_omega_x = np.zeros(N)
true_omega_x = omega_x * np.ones(N)

fidelity = np.zeros(N)

# ============================================================
# Convert mean spin vector -> spin coherent state
# ============================================================

from scipy.special import comb


def coherent_state_from_bloch(Jvec):

    norm = np.linalg.norm(Jvec)

    if norm < 1e-10:
        theta = 0.0
        phi_angle = 0.0
    else:
        nx, ny, nz = Jvec / norm
        theta = np.arccos(np.clip(nz, -1, 1))
        phi_angle = np.arctan2(ny, nx)

    coeffs = np.zeros(dim, dtype=complex)

    for idx, m in enumerate(m_vals):

        k = int(j - m)

        coeff = np.sqrt(comb(int(2 * j), k))

        coeff *= (
            np.cos(theta / 2) ** (int(2 * j) - k)
            * (np.sin(theta / 2) * np.exp(1j * phi_angle)) ** k
        )

        coeffs[idx] = coeff

    coeffs /= np.linalg.norm(coeffs)

    return coeffs

# ============================================================
# Kalman filter loop
# ============================================================

for k, t in enumerate(time):

    # Current parameter estimate
    omega_x_est = x_est[3]

    # Effective magnetic field using estimated parameter
    Bx = Omega * np.cos(phi(t)) + omega_x_est
    By = Omega * np.sin(phi(t)) + omega_y
    Bz = omega_z

    # Current spin estimate
    Jx_est, Jy_est, Jz_est = x_est[:3]

    # --------------------------------------------------------
    # Linearized augmented dynamics
    # --------------------------------------------------------
    #
    # d<J>/dt = B x <J>
    #
    # omega_x is modeled as a constant parameter:
    #
    # d omega_x / dt = 0
    #
    # The Jacobian contains the sensitivity of the spin
    # dynamics to omega_x.
    #
    # --------------------------------------------------------

    A = np.array([
        [0.0, -Bz, By, 0.0],
        [Bz, 0.0, -Bx, -Jz_est],
        [-By, Bx, 0.0, Jy_est],
        [0.0, 0.0, 0.0, 0.0]
    ])

    # Discrete-time transition matrix
    F = np.eye(4) + dt * A

    # --------------------------------------------------------
    # Prediction
    # --------------------------------------------------------

    x_pred = F @ x_est

    # Nonlinear deterministic update
    Jx_dot = By * Jz_est - Bz * Jy_est
    Jy_dot = Bz * Jx_est - Bx * Jz_est
    Jz_dot = Bx * Jy_est - By * Jx_est

    x_pred[0] = Jx_est + dt * Jx_dot
    x_pred[1] = Jy_est + dt * Jy_dot
    x_pred[2] = Jz_est + dt * Jz_dot

    # omega_x assumed constant
    x_pred[3] = omega_x_est

    P_pred = F @ P @ F.T + Q

    # --------------------------------------------------------
    # Measurement update
    # --------------------------------------------------------

    y = np.array([[measurements[k]]])

    innovation = y - H_meas @ x_pred.reshape(-1, 1)

    S = H_meas @ P_pred @ H_meas.T + R

    K = P_pred @ H_meas.T @ np.linalg.inv(S)

    x_est = x_pred + (K @ innovation).flatten()

    P = (np.eye(3) - K @ H_meas) @ P_pred

    estimated_means[k] = x_est[:3]
    estimated_omega_x[k] = x_est[3]

    # --------------------------------------------------------
    # Fidelity estimate
    # --------------------------------------------------------

    psi_est = coherent_state_from_bloch(x_est)

    psi_true = true_states[k]

    fidelity[k] = np.abs(np.vdot(psi_est, psi_true)) ** 2

# ============================================================
# Plot trajectories
# ============================================================

fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)

labels = [r'$\langle J_x \rangle$',
          r'$\langle J_y \rangle$',
          r'$\langle J_z \rangle$']

for i in range(3):

    axes[i].plot(time, true_means[:, i], label='True', linewidth=2)

    axes[i].plot(time,
                 estimated_means[:, i],
                 '--',
                 label='Estimated',
                 linewidth=2)

    axes[i].set_ylabel(labels[i])
    axes[i].grid(True)
    axes[i].legend()

axes[-1].set_xlabel('Time')

plt.tight_layout()
plt.show()

# ============================================================
# Plot omega_x estimation
# ============================================================

plt.figure(figsize=(10, 4))

plt.plot(time,
         true_omega_x,
         linewidth=2,
         label='True $\omega_x$')

plt.plot(time,
         estimated_omega_x,
         '--',
         linewidth=2,
         label='Estimated $\omega_x$')

plt.xlabel('Time')
plt.ylabel(r'$\omega_x$')
plt.title(r'Parameter Estimation of $\omega_x$')
plt.grid(True)
plt.legend()

plt.show()

# ============================================================
# Plot fidelity
# ============================================================

plt.figure(figsize=(10, 4))

plt.plot(time, fidelity, linewidth=2)

plt.xlabel('Time')
plt.ylabel('Fidelity')
plt.title('Estimated-State Fidelity')
plt.grid(True)

plt.show()
```

---

# Interpretation

The filter estimates the spin vector using:

* The deterministic precession model
* Noisy observations of only (\langle J_z \rangle)

Even though only one observable is measured, the coupled spin dynamics allow the Kalman filter to infer the transverse components (\langle J_x \rangle) and (\langle J_y \rangle).

The fidelity plot compares:

* The exact quantum state
* A spin coherent state reconstructed from the estimated Bloch vector

Because the true dynamics remain close to a coherent-state manifold under linear spin rotations, the fidelity is typically high when the filter tracks the trajectory accurately.

---

# Notes

## 1. Why the dynamics are linear

For a Hamiltonian linear in spin operators,

[
H = \mathbf{B}(t) \cdot \mathbf{J}
]

the expectation values satisfy the classical precession equation

[
\frac{d}{dt}\langle \mathbf{J} \rangle
======================================

\mathbf{B}(t) \times \langle \mathbf{J} \rangle
]

which is exactly linear in the state vector.

## 2. Why this is only an approximate quantum filter

The Kalman filter operates only on first moments:

[
(\langle J_x \rangle, \langle J_y \rangle, \langle J_z \rangle)
]

and does not propagate the full density matrix.

The fidelity calculation therefore reconstructs an approximate spin coherent state from the estimated Bloch vector.

## 3. Including measurement backaction

If you wanted to include quantum measurement backaction, you would instead evolve:

* A stochastic master equation (SME)
* Or a stochastic Schrödinger equation (SSE)

and the estimation problem would become nonlinear.

In that case:

* An Extended Kalman Filter (EKF)
* Unscented Kalman Filter (UKF)
* Cubature Kalman Filter (CKF)
* Particle filter

would generally be more appropriate.
